In [ ]:

!pip install qiskit==1.1.1 qiskit-aer==0.15.1 qiskit-machine-learning==0.7.2 qiskit-ibm-runtime==0.24.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.4/133.4 kB 3.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.8/97.8 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.8/386.8 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 81.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.3/50.3 MB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 8.8 MB/s eta 0:00:00
  Created whe

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import ShuffleSplit
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from qiskit.quantum_info import Statevector
from qiskit import QuantumCircuit
import warnings
# Suppress potential Qiskit/sklearn warnings for cleaner output
warnings.filterwarnings('ignore')
# --- Constants & Setup ---
N_SPLITS = 20
TEST_SIZE = 0.20
BANDWIDTH_OPTIONS = np.round(np.arange(0.1, 1.1, 0.1), 1)
# --- Load Data ---
df = pd.read_csv("data.csv")
y = (df.iloc[:, -1].str.strip() == "P").astype(int).values
X_raw = df.iloc[:, 1:-1].values.astype(float)
# --- 9-Qubit Circuit (18 params for 18 task features) ---
def sv_9q(x, b):
    qc = QuantumCircuit(9)
    # Layer 1: Rx rotations (first 9 features)
    for q in range(9): qc.rx(b * x[q], q)
    # Entangling Layer (CZ Ring)
    for q in range(8): qc.cz(q, q+1)
    qc.cz(8, 0)
    # Layer 2: Ry rotations (next 9 features)
    for q in range(9): qc.ry(b * x[9+q], q)
    # Entangling Layer (CX Ring)
    for q in range(8): qc.cx(q, q+1)
    qc.cx(8, 0)
    return Statevector(qc).data
def get_kernel(A, B, b):
  sa = np.array([sv_9q(x, b) for x in A])
  sb = np.array([sv_9q(x, b) for x in B])
  return np.abs(sa @ sb.conj().T) ** 2
# --- Experiment Loop ---
ss = ShuffleSplit(n_splits=N_SPLITS, test_size=TEST_SIZE, random_state=42)
q_all_25_list, q_best_5_list = [], []
for split_num, (train_val_idx, test_idx) in enumerate(ss.split(y)):
    print(f"split{split_num + 1}/{N_SPLITS}")
    # 60/20/20 split logic
    split_point = int(len(train_val_idx) * 0.75)
    train_idx = train_val_idx[:split_point]
    val_idx = train_val_idx[split_point:]
    y_tr, y_val, y_te = y[train_idx], y[val_idx], y[test_idx]
    all_task_preds, task_val_accs = [], []
    for t in range(25):
        X_task = X_raw[:, t*18:(t+1)*18]
        # DOUBLE STANDARDIZATION
        X_s = StandardScaler().fit_transform(X_task)
        X_final = StandardScaler().fit_transform(X_s)
        X_tr_task = X_final[train_idx]
        X_val_task = X_final[val_idx]
        X_te_task = X_final[test_idx]
        # Tune bandwidth (b) on Validation
        best_b, best_v_acc = 0.4, -1
        for b in BANDWIDTH_OPTIONS:
            K_tr_v = get_kernel(X_tr_task, X_tr_task, b)
            K_val_v = get_kernel(X_val_task, X_tr_task, b)
            clf_v = SVC(kernel="precomputed", C=1.0).fit(K_tr_v, y_tr)
            v_acc = accuracy_score(y_val, clf_v.predict(K_val_v))
            if v_acc > best_v_acc:
                best_v_acc, best_b = v_acc, b
        # Final prediction
        X_comb = np.vstack((X_tr_task, X_val_task))
        y_comb = np.hstack((y_tr, y_val))
        K_final_tr = get_kernel(X_comb, X_comb, best_b)
        K_final_te = get_kernel(X_te_task, X_comb, best_b)
        final_clf = SVC(kernel="precomputed", C=1.0).fit(K_final_tr, y_comb)
        task_test_preds = final_clf.predict(K_final_te)
        all_task_preds.append(task_test_preds)
        task_val_accs.append(best_v_acc)
        print(f"  Task {t+1:02d}: Test Acc={accuracy_score(y_te, task_test_preds)*100:.1f}%, b={best_b}")
    # --- Vote logic ---
    def get_voted_acc(preds_list, true_labels):
        arr = np.array(preds_list)
        voted_labels = [np.bincount(arr[:, i]).argmax() for i in range(arr.shape[1])]
        return accuracy_score(true_labels, voted_labels) * 100
    s_all_25 = get_voted_acc(all_task_preds, y_te)
    q_all_25_list.append(s_all_25)
    best_5_indices = np.argsort(task_val_accs)[-5:]
    s_best_5 = get_voted_acc([all_task_preds[i] for i in best_5_indices], y_te)
    q_best_5_list.append(s_best_5)
    print(f">> Split {split_num+1} Result: All 25 = {s_all_25:.2f}%, Best 5 = {s_best_5:.2f}%\n")
# 1. Report Accuracies
print(f"'All 25' Accuracy(Majority Vote Decision): {np.mean(q_all_25_list):.2f}% ± {np.std(q_all_25_list):.2f}%")
print(f"'Best 5' Accuracy(Majority Vote Decision):  {np.mean(q_best_5_list):.2f}% ± {np.std(q_best_5_list):.2f}%")


split1/20
  Task 01: Test Acc=54.3%, b=0.4
  Task 02: Test Acc=74.3%, b=0.4
  Task 03: Test Acc=74.3%, b=0.2
  Task 04: Test Acc=57.1%, b=0.1
  Task 05: Test Acc=62.9%, b=0.1
  Task 06: Test Acc=74.3%, b=0.1
  Task 07: Test Acc=65.7%, b=0.6
  Task 08: Test Acc=57.1%, b=0.8
  Task 09: Test Acc=65.7%, b=0.1
  Task 10: Test Acc=77.1%, b=0.3
  Task 11: Test Acc=68.6%, b=0.1
  Task 12: Test Acc=68.6%, b=1.0
  Task 13: Test Acc=65.7%, b=0.5
  Task 14: Test Acc=68.6%, b=0.6
  Task 15: Test Acc=71.4%, b=0.1
  Task 16: Test Acc=85.7%, b=0.4
  Task 17: Test Acc=80.0%, b=0.3
  Task 18: Test Acc=71.4%, b=0.4
  Task 19: Test Acc=68.6%, b=0.7
  Task 20: Test Acc=74.3%, b=0.8
  Task 21: Test Acc=71.4%, b=0.4
  Task 22: Test Acc=65.7%, b=0.5
  Task 23: Test Acc=80.0%, b=0.4
  Task 24: Test Acc=74.3%, b=0.6
  Task 25: Test Acc=77.1%, b=0.2
>> Split 1 Result: All 25 = 82.86%, Best 5 = 77.14%

split2/20
  Task 01: Test Acc=54.3%, b=0.3
  Task 02: Test Acc=65.7%, b=0.4
  Task 03: Test Acc=65.7%, b=0.1
  T